# Détection de pose sur Stanford Dogs

Ce notebook applique la détection de pose (keypoints) sur des images du dataset Stanford Dogs en utilisant :
- **OpenCV DNN** avec le modèle pre-entraîné MPII / COCO
- **scikit-image** pour l'affichage et le post-traitement des keypoints
- Visualisation des squelettes et heatmaps de confiance

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/home/SPeillet/bcs_determination")

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
import random
import urllib.request
import json

import cv2

print(f"OpenCV: {cv2.__version__}")
print(f"CUDA devices: {cv2.cuda.getCudaEnabledDeviceCount() if hasattr(cv2, 'cuda') else 'N/A'}")

## 1. Téléchargement du modèle OpenPose (MPII)

In [ ]:
MODEL_DIR = ROOT / "models" / "pose"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PROTO_PATH = MODEL_DIR / "pose_deploy_linevec_faster_4_stages.prototxt"
WEIGHTS_PATH = MODEL_DIR / "pose_iter_160000.caffemodel"

HAS_OPENPOSE = PROTO_PATH.exists() and WEIGHTS_PATH.exists()

if not HAS_OPENPOSE:
    urls_proto = [
        "https://raw.githubusercontent.com/CMU-Perceptual-Computing-Lab/openpose/master/models/pose/mpi/pose_deploy_linevec_faster_4_stages.prototxt",
        "https://www.vision.ai/files/pose_deploy_linevec_faster_4_stages.prototxt",
    ]
    urls_weights = [
        "https://www.vision.ai/files/pose_iter_160000.caffemodel",
        "https://datasets.vision.ai/pose/mpi/pose_iter_160000.caffemodel",
    ]
    
    for url in urls_proto:
        try:
            print(f"Trying: {url}")
            urllib.request.urlretrieve(url, str(PROTO_PATH))
            print(f"  Saved: {PROTO_PATH}")
            break
        except Exception as e:
            print(f"  Failed: {e}")
    
    for url in urls_weights:
        try:
            print(f"Trying: {url}")
            urllib.request.urlretrieve(url, str(WEIGHTS_PATH))
            print(f"  Saved: {WEIGHTS_PATH}")
            break
        except Exception as e:
            print(f"  Failed: {e}")
    
    HAS_OPENPOSE = PROTO_PATH.exists() and WEIGHTS_PATH.exists()

if HAS_OPENPOSE:
    print("OpenPose MPII model ready!")
else:
    print("WARNING: Could not download OpenPose model. OpenPose sections will be skipped.")
    print("You can manually download from:")
    print("  https://github.com/CMU-Perceptual-Computing-Lab/openpose/tree/master/models/pose/mpi")

In [ ]:
MPII_KEYPOINTS = [
    "Head",           # 0
    "Neck",           # 1
    "R.Shoulder",     # 2
    "R.Elbow",        # 3
    "R.Wrist",        # 4
    "L.Shoulder",     # 5
    "L.Elbow",        # 6
    "L.Wrist",        # 7
    "R.Hip",          # 8
    "R.Knee",         # 9
    "R.Ankle",        # 10
    "L.Hip",          # 11
    "L.Knee",         # 12
    "L.Ankle",        # 13
    "Chest",          # 14
    "Background",     # 15
]

MPII_SKELETON = [
    (0, 1),   # Head -> Neck
    (1, 2),   # Neck -> R.Shoulder
    (1, 5),   # Neck -> L.Shoulder
    (2, 3),   # R.Shoulder -> R.Elbow
    (3, 4),   # R.Elbow -> R.Wrist
    (5, 6),   # L.Shoulder -> L.Elbow
    (6, 7),   # L.Elbow -> L.Wrist
    (1, 14),  # Neck -> Chest
    (14, 8),  # Chest -> R.Hip
    (14, 11), # Chest -> L.Hip
    (8, 9),   # R.Hip -> R.Knee
    (9, 10),  # R.Knee -> R.Ankle
    (11, 12), # L.Hip -> L.Knee
    (12, 13), # L.Knee -> L.Ankle
]

SKELETON_COLORS = plt.cm.Set1(np.linspace(0, 1, len(MPII_SKELETON)))

NUM_KEYPOINTS = len(MPII_KEYPOINTS)

## 2. Chargement des images

In [ ]:
DATA_DIR = ROOT / "data" / "stanford_dogs" / "Images"
SEED = 42
random.seed(SEED)

all_images = sorted(glob(str(DATA_DIR / "*" / "*.jpg")))
sample_paths = random.sample(all_images, 6)

def load_rgb(path, max_size=600):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size), Image.LANCZOS)
    return np.array(img)

def get_breed(path):
    return "-".join(Path(path).parent.name.split("-")[1:])

fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 4))
for ax, p in zip(axes, sample_paths):
    ax.imshow(load_rgb(p))
    ax.set_title(get_breed(p), fontsize=9)
    ax.axis("off")
fig.suptitle("Samples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Inférence OpenPose (OpenCV DNN)

In [ ]:
def detect_pose(image_rgb, proto_path, weights_path, threshold=0.1):
    net = cv2.dnn.readNetFromCaffe(str(proto_path), str(weights_path))
    
    h, w = image_rgb.shape[:2]
    inp_h = 368
    inp_w = max(368, int(w * 368 / h))
    
    blob = cv2.dnn.blobFromImage(image_rgb, 1.0 / 255, (inp_w, inp_h), (0, 0, 0), swapRB=False, crop=False)
    net.setInput(blob)
    output = net.forward()
    
    out_h, out_w = output.shape[2], output.shape[3]
    
    keypoints = []
    for i in range(NUM_KEYPOINTS):
        confidence_map = output[0, i, :, :]
        _, conf, _, point = cv2.minMaxLoc(confidence_map)
        
        x = int(point[0] * w / out_w)
        y = int(point[1] * h / out_h)
        
        keypoints.append({
            "name": MPII_KEYPOINTS[i],
            "x": x, "y": y,
            "confidence": conf,
            "detected": conf > threshold,
        })
    
    heatmaps = output[0, :NUM_KEYPOINTS, :, :]  # (K, H', W')
    
    return keypoints, heatmaps


if HAS_OPENPOSE:
    print("Model loaded successfully!")
else:
    print("WARNING: Model files not found. OpenPose sections will be skipped.")

## 4. Visualisation des poses détectées

In [ ]:
def draw_pose(image_rgb, keypoints, skeleton=MPII_SKELETON, point_radius=6, line_width=3):
    canvas = image_rgb.copy()
    
    for part_a, part_b in skeleton:
        kp_a = keypoints[part_a]
        kp_b = keypoints[part_b]
        if kp_a["detected"] and kp_b["detected"]:
            color = (SKELETON_COLORS[MPII_SKELETON.index((part_a, part_b))][:3] * 255).astype(int).tolist()
            cv2.line(canvas, (kp_a["x"], kp_a["y"]), (kp_b["x"], kp_b["y"]), color, line_width, cv2.LINE_AA)
    
    for kp in keypoints:
        if kp["detected"] and kp["name"] != "Background":
            conf_color = (0, int(255 * kp["confidence"]), int(255 * (1 - kp["confidence"])))
            cv2.circle(canvas, (kp["x"], kp["y"]), point_radius, conf_color, -1, cv2.LINE_AA)
            cv2.circle(canvas, (kp["x"], kp["y"]), point_radius, (255, 255, 255), 1, cv2.LINE_AA)
    
    return canvas

if HAS_OPENPOSE:
    for p in sample_paths:
        img = load_rgb(p)
        keypoints, heatmaps = detect_pose(img, PROTO_PATH, WEIGHTS_PATH, threshold=0.1)
        
        annotated = draw_pose(img, keypoints)
        
        n_detected = sum(1 for kp in keypoints if kp["detected"] and kp["name"] != "Background")
        mean_conf = np.mean([kp["confidence"] for kp in keypoints if kp["detected"]])
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(img)
        axes[0].set_title(f"Original — {get_breed(p)}")
        axes[0].axis("off")
        
        axes[1].imshow(annotated)
        axes[1].set_title(f"Pose ({n_detected}/{NUM_KEYPOINTS-1} keypoints, conf={mean_conf:.2f})")
        axes[1].axis("off")
        
        plt.tight_layout()
        plt.show()
        
        for kp in keypoints:
            if kp["detected"] and kp["name"] != "Background":
                print(f"  {kp['name']:15s}  ({kp['x']:4d}, {kp['y']:4d})  conf={kp['confidence']:.3f}")
        print()
else:
    print("OpenPose model not available. Skipping.")

## 5. Heatmaps de confiance

In [ ]:
if HAS_OPENPOSE:
    for p in sample_paths[:3]:
        img = load_rgb(p)
        keypoints, heatmaps = detect_pose(img, PROTO_PATH, WEIGHTS_PATH, threshold=0.1)
        
        n_kp = sum(1 for kp in keypoints if kp["detected"] and kp["name"] != "Background")
        
        cols = 4
        rows = (NUM_KEYPOINTS + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        axes_flat = axes.flatten()
        
        for i in range(NUM_KEYPOINTS):
            hm = heatmaps[i]
            hm_resized = cv2.resize(hm, (img.shape[1], img.shape[0]))
            
            axes_flat[i].imshow(img, alpha=0.4)
            axes_flat[i].imshow(hm_resized, cmap="jet", alpha=0.6)
            
            kp = keypoints[i]
            detected = kp["detected"] and kp["name"] != "Background"
            title_color = "green" if detected else "red"
            axes_flat[i].set_title(f"{MPII_KEYPOINTS[i]}\n{kp['confidence']:.2f}", fontsize=8, color=title_color)
            axes_flat[i].axis("off")
        
        for j in range(NUM_KEYPOINTS, len(axes_flat)):
            axes_flat[j].axis("off")
        
        fig.suptitle(f"Confidence Heatmaps — {get_breed(p)} ({n_kp} keypoints)", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

## 6. Alternative : MediaPipe Pose

MediaPipe offre une détection de pose plus moderne et souvent plus robuste que l'OpenPose MPII.

In [ ]:
try:
    import mediapipe as mp
    HAS_MEDIAPIPE = True
    print(f"MediaPipe version: {mp.__version__}")
except ImportError:
    HAS_MEDIAPIPE = False
    print("MediaPipe not installed. Install with: pip install mediapipe")
    print("Skipping MediaPipe section.")

In [ ]:
if HAS_MEDIAPIPE:
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    mp_styles = mp.solutions.drawing_styles

    fig, axes = plt.subplots(len(sample_paths), 2, figsize=(14, 7 * len(sample_paths)))
    if len(sample_paths) == 1:
        axes = axes.reshape(1, -1)

    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,
        min_detection_confidence=0.3,
        enable_segmentation=False,
    ) as pose:
        for row, p in enumerate(sample_paths):
            img = load_rgb(p)
            results = pose.process(img)

            axes[row, 0].imshow(img)
            axes[row, 0].set_title(f"Original — {get_breed(p)}")
            axes[row, 0].axis("off")

            annotated = img.copy()
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated,
                    results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
                )
                n_lm = len(results.pose_landmarks.landmark)
                axes[row, 1].set_title(f"MediaPipe Pose ({n_lm} landmarks)")
            else:
                axes[row, 1].set_title("MediaPipe Pose — No detection")

            axes[row, 1].imshow(annotated)
            axes[row, 1].axis("off")

    plt.suptitle("MediaPipe Pose Detection", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

## 7. Alternative : YOLOv8-Pose (Ultralytics)

In [ ]:
try:
    from ultralytics import YOLO
    HAS_YOLO = True
    print("Ultralytics YOLO available")
except ImportError:
    HAS_YOLO = False
    print("Ultralytics not installed. Install with: pip install ultralytics")
    print("Skipping YOLO-Pose section.")

In [ ]:
if HAS_YOLO:
    yolo_model = YOLO("yolov8n-pose.pt")

    fig, axes = plt.subplots(len(sample_paths), 2, figsize=(14, 7 * len(sample_paths)))
    if len(sample_paths) == 1:
        axes = axes.reshape(1, -1)

    for row, p in enumerate(sample_paths):
        img = load_rgb(p)
        results = yolo_model(img, verbose=False)
        annotated = results[0].plot()

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f"Original — {get_breed(p)}")
        axes[row, 0].axis("off")

        n_persons = len(results[0].boxes)
        axes[row, 1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[row, 1].set_title(f"YOLOv8-Pose ({n_persons} detections)")
        axes[row, 1].axis("off")

    plt.suptitle("YOLOv8-Pose Detection", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

## 8. Comparaison des méthodes

In [ ]:
methods_available = ["OpenPose (MPII)"]
if HAS_MEDIAPIPE:
    methods_available.append("MediaPipe Pose")
if HAS_YOLO:
    methods_available.append("YOLOv8-Pose")

print("Methodes de detection de pose disponibles :")
for m in methods_available:
    print(f"  - {m}")

print(f"\nNote : Ces modeles sont entraines sur des poses HUMAINES.")
print(f"Sur des images de chiens, les resultats seront limitees.")
print(f"Pour une detection de pose specifique aux animaux,")
print(f"il faudrait utiliser un modele entraine sur AnimalPose ou AP-10K.")